# Experiment II — extrapolation toward the LINEAR limit (ε → 1)

**The family parameter ε.** Wilkening's standing-wave branch is naturally parameterized by
the downward **crest acceleration** $A_c$ (column 4 of the `parameters` table; Mercer &
Roberts 1992, Wilkening PRL 2011). We use its distance from free fall,

$$\varepsilon \;=\; 1 - A_c/g \;=\; 1 - A_c \quad (g = 1),$$

so that ε → 0 is the singular free-fall (steep) limit and ε → 1 the linear limit. ε is
monotone along the branch (unlike the wave height h, which folds at `idx 209`), and smaller
ε means a steeper crest. Across the dataset: ε = 0.992 (V101, near-linear) down to
ε = 0.0123 (V240, max slope 1.13).

**This notebook's split:**
- **Training**: the large-amplitude side, `idx 111..240` minus the ID holdouts
  (training ε ∈ [0.0123, 0.904])
- **ID tests**: `idx {125, 150, 175, 200, 225}` (ε ∈ [0.039, 0.76]), each bracketed by trained neighbors
- **OOD block**: the gentle end `idx 101..110`, ε ∈ [0.913, 0.992] — strictly ABOVE the
  training maximum, so a genuine range extrapolation toward the linear limit, where the wave
  amplitude vanishes like 1 − ε

**Key hyperparameters**: FNO with `MODES = 64`, `WIDTH = 64`, `LAYERS = 8`; input channels
$[\eta_0(x), \tau, \cos x, \sin x]$ on the 512-point mirrored grid; τ = t/(T/4); every
2nd stored timeslice (101 τ's per solution); channelwise standardization by TRAINING
statistics; relative $L^2$ errors over the space–time quarter period.

**Folder layout**: raw data is READ from `jw_standing_wave_data_only`; everything this
notebook writes goes to `jw_standing_wave_fno` (npy cache, model checkpoint), with Python
figures in `figures-python/` and all `.mat` exports for MATLAB in `figures-matlab/`.


# Fourier Neural Operator for the Standing-Wave Family 

**Idea.** Learn the **solution operator**
of the free-surface water-wave equations directly:

$$\mathcal{G}_\theta:\;\big(\eta_0(x),\,t\big)\;\longmapsto\;\big(\eta,\varphi,\mu\big)(x,t),\qquad t\in[0,\,T/4].$$

At $t=0$ the wave is at rest ($\varphi=\mu=0$ in the V files), so the initial elevation
$\eta_0(x)$ alone determines the whole solution — a clean operator-learning setup.

**Why FNO is a natural fit here**
* The domain is $2\pi$-periodic in $x$ and the grid is uniform → spectral convolutions
  (FFT-based kernels) match the geometry exactly, with no boundary handling needed.
* The solutions form a smooth one-parameter family (indexed by wave height $\nu$), so a single
  operator can interpolate across the family from a handful of training solutions.
* Time enters only as a scalar conditioning channel (normalized by each solution's own
  quarter-period $T/4$), so one network predicts any time slice in one forward pass —
  no autoregressive rollout, no error accumulation.

**Caveats to keep in mind**
* Near-limiting waves (large idx) develop sharp crests → significant high-wavenumber content.
  The spectral truncation (`MODES`) must be generous. **Split used here (REVERSED experiment):** training on the
  large-$\nu$ (large-amplitude) side of the family, OOD test on the small-$\nu$ gentle end
  (V101–V110, toward the linear regime), plus scattered interior ID holdouts.
  This is the mirror image of the previous experiment (train small $\nu$ → OOD large $\nu$);
  comparing the two OOD errors tests whether extrapolation difficulty tracks crest
  steepening or is symmetric in $\nu$-distance.
* This is purely data-driven: the network is *not* constrained by the water-wave PDE.
  A physics-informed loss (kinematic/dynamic boundary conditions) could be added later.

**Data.** Jon Wilkening's `V101..V240` files: each stores one quarter solution,
$0\le t\le T/4$, $0\le x\le\pi$, 201 timeslices $\times$ 257 points, columns $\eta,\varphi,\mu$.
All three fields are even in $x$, so a plain mirror extension gives the full periodic
512-point grid on $(-\pi,\pi]$ (same convention as `full.pl`).

In [ ]:
# =========================
# Imports and configuration
# =========================
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- Paths ----
DATA_DIR = os.environ.get("STANDING_WAVE_DATA", "../data")   # raw V files + parameters
OUT_DIR  = ".."                                                # repository root
MAT_DIR  = os.path.join(OUT_DIR, "results")       # .mat exports
CACHE_DIR = os.path.join(OUT_DIR, "cache")       # .npy cache
for _d in (MAT_DIR, CACHE_DIR):
    os.makedirs(_d, exist_ok=True)

# ---- Solution split: train on LARGE nu, test OOD on SMALL nu (REVERSED case) ----
# Mirror image of the previous experiment. The training set now covers the
# large-amplitude (large-nu) side of the standing-wave family, V111..V240
# (minus the ID holdouts). The OOD test set is the SMALL-nu, small-amplitude
# end, V101..V110 -- the gentlest, nearly-linear waves. The experiment asks:
# trained on moderate-to-steep waves, can the operator extrapolate DOWN to the
# nearly-linear regime? (Physically this direction should be easier than
# extrapolating toward the limiting wave: small-nu profiles are smooth,
# near-sinusoidal, with fast spectral decay.)
#   * TEST_ID_INDICES : scattered interior solutions on the trained large-nu
#                       side, each surrounded by trained neighbors -> ID
#   * TEST_OOD_INDICES: the entire small-nu end, held out as a block -> OOD
# NOTE: unlike the previous (large-nu OOD) case, here the nu-range test and the
# index-block definition AGREE: the OOD block has nu = 0.008..0.080, cleanly
# below the training minimum nu = 0.088. The split is still defined by the
# index block for consistency with the previous experiment.
ALL_INDICES      = list(range(101, 241))          # V101 .. V240
TEST_ID_INDICES  = [125, 150, 175, 200, 225]      # interior holdouts (large-nu side)
TEST_OOD_INDICES = list(range(101, 111))          # small-nu gentle end, held out entirely
TEST_INDICES     = TEST_ID_INDICES + TEST_OOD_INDICES
TRAIN_INDICES    = [i for i in ALL_INDICES if i not in TEST_INDICES]
print(f"{len(TRAIN_INDICES)} training / {len(TEST_INDICES)} test solutions")

# ---- Data resolution ----
TIME_STRIDE = 2      # keep every 2nd of the 201 timeslices (set 1 for full resolution)

# ---- FNO hyperparameters ----
MODES  = 64          # retained Fourier modes (out of 257 rfft modes on the 512-grid)
WIDTH  = 64          # channel width of the hidden layers
LAYERS = 8          # number of Fourier blocks

# ---- Training hyperparameters ----
EPOCHS       = 1000
BATCH_SIZE   = 64
LR           = 1e-3
WEIGHT_DECAY = 1e-6
PRINT_EVERY  = 25
SEED         = 0

# ---- Device ----
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = get_device()
torch.manual_seed(SEED); np.random.seed(SEED)
print("Device:", DEVICE)

## 1. Load the raw `V` files

Each `V<idx>` file: 201 timeslices $\times$ 257 rows $\times$ 3 columns ($\eta,\varphi,\mu$),
slices separated by blank lines (ignored by `np.loadtxt`). The `parameters` table provides the
period $T$ (col 1) and the family parameter — wave height (col 2) — per solution.

In [ ]:
# def load_V(path, n_t=201, n_x=257):
#     """Read a quarter-solution file -> (eta, phi, mu), each (n_t, n_x)."""
#     A = np.loadtxt(path)                      # (n_t * n_x, 3); blank lines skipped
#     if A.shape != (n_t * n_x, 3):
#         raise ValueError(f"{path}: expected {(n_t * n_x, 3)}, got {A.shape}")
#     A = A.reshape(n_t, n_x, 3)                # slice-major, matches file order
#     return A[..., 0], A[..., 1], A[..., 2]
def load_V(path, n_t=201, n_x=257):
    """Read a quarter-solution file -> (eta, phi, mu), each (n_t, n_x).
    Caches the parsed array as .npy: parsing 130+ text files with np.loadtxt
    takes minutes, the cached reload takes seconds."""
    cache = os.path.join(CACHE_DIR, os.path.basename(path) + ".npy")
    if os.path.exists(cache):
        A = np.load(cache)
    else:
        A = np.loadtxt(path).reshape(n_t, n_x, 3)
        np.save(cache, A)
    return A[..., 0], A[..., 1], A[..., 2]

def mirror_extend(A):
    """Even extension along x: [0, pi] (257 pts) -> (-pi, pi] (512 uniform pts).
    Valid because eta, phi, mu are all even in x (see full.pl: j = |M - i|,
    no sign flip for the spatial reflection)."""
    return np.concatenate([A[..., -2:0:-1], A], axis=-1)

# Parameters table: idx | period T | wave_height | crest_accel | max_slope | ||eta_x||^2
P_table = np.loadtxt(os.path.join(DATA_DIR, "parameters"))
params  = {int(r[0]): r for r in P_table}

# Full periodic x-grid on (-pi, pi], uniform, spacing pi/256 -> ideal for the FFT in FNO
x_quarter = np.linspace(0.0, np.pi, 257)
x_grid    = np.concatenate([-x_quarter[-2:0:-1], x_quarter])          # (512,)
NX        = x_grid.size
print("Periodic grid:", NX, "points,  dx =", x_grid[1] - x_grid[0])

## 2. Assemble the operator-learning dataset

One sample = (initial elevation $\eta_0(x)$, normalized time $t\in[0,1]$) $\to$ snapshot
$(\eta,\varphi,\mu)(x,t)$. Time is normalized by each solution's own quarter period, so all
solutions share one conditioning variable; the physical period is kept for post-processing.

In [ ]:
def build_dataset(indices):
    """Stack (eta0, t) -> (eta, phi, mu) samples for a list of solution indices.

    Returns
    -------
    X0     : (B, NX)      initial elevation eta0(x), repeated across the times of its solution
    Tn     : (B,)         normalized time in [0, 1] (fraction of the stored quarter period)
    Y      : (B, 3, NX)   target snapshot, channels = [eta, phi, mu]
    sol_id : (B,)         which solution index each sample came from (for per-solution metrics)
    """
    X0, Tn, Y, sol_id = [], [], [], []
    t_full = np.linspace(0.0, 1.0, 201)[::TIME_STRIDE]
    for idx in indices:
        eta, phi, mu = load_V(os.path.join(DATA_DIR, f"V{idx}"))
        eta = mirror_extend(eta)[::TIME_STRIDE]     # (Mt, 512)
        phi = mirror_extend(phi)[::TIME_STRIDE]
        mu  = mirror_extend(mu)[::TIME_STRIDE]
        eta0 = eta[0]                               # phi = mu = 0 at t = 0 (wave at rest)
        for j in range(eta.shape[0]):
            X0.append(eta0)
            Tn.append(t_full[j])
            Y.append(np.stack([eta[j], phi[j], mu[j]], axis=0))
            sol_id.append(idx)
        print(f"V{idx}: nu = {params[idx][2]:.4f},  T = {params[idx][1]:.4f},  {eta.shape[0]} timeslices")
    return (np.asarray(X0, np.float32), np.asarray(Tn, np.float32),
            np.asarray(Y, np.float32), np.asarray(sol_id))

print("--- training solutions ---")
X0_tr, T_tr, Y_tr, id_tr = build_dataset(TRAIN_INDICES)
print("--- test solutions ---")
X0_te, T_te, Y_te, id_te = build_dataset(TEST_INDICES)
print("train samples:", X0_tr.shape[0], " test samples:", X0_te.shape[0])

In [ ]:
# ---- Channel-wise normalization (statistics from the TRAINING set only) ----
# eta / phi / mu live on very different scales; standardize each output channel,
# and normalize the eta0 input channel with the eta statistics.
mu_y = Y_tr.mean(axis=(0, 2), keepdims=True)          # (1, 3, 1)
sd_y = Y_tr.std(axis=(0, 2), keepdims=True) + 1e-8    # (1, 3, 1)

cosx = np.cos(x_grid).astype(np.float32)              # periodic coordinate embedding:
sinx = np.sin(x_grid).astype(np.float32)              # (cos x, sin x) instead of raw x

def pack_inputs(X0, Tn):
    """Build the (B, 4, NX) input tensor with channels:
       [ normalized eta0(x),  t (broadcast),  cos x,  sin x ]."""
    B = X0.shape[0]
    e0 = (X0 - mu_y[0, 0]) / sd_y[0, 0]
    ch = np.stack([e0,
                   np.repeat(Tn[:, None], NX, axis=1),
                   np.tile(cosx, (B, 1)),
                   np.tile(sinx, (B, 1))], axis=1)
    return torch.tensor(ch)

X_tr = pack_inputs(X0_tr, T_tr)
X_te = pack_inputs(X0_te, T_te)
Yn_tr = torch.tensor((Y_tr - mu_y) / sd_y)            # normalized targets
Yn_te = torch.tensor((Y_te - mu_y) / sd_y)
print("input tensor:", tuple(X_tr.shape), " target tensor:", tuple(Yn_tr.shape))

## 3. The 1D Fourier Neural Operator

Standard FNO block: a spectral convolution (learned complex multipliers on the lowest
`MODES` Fourier modes) in parallel with a pointwise linear channel mix, followed by GELU.
The periodic geometry of the wave data is exactly what the FFT assumes, so no padding or
boundary tricks are needed.

In [ ]:
class SpectralConv1d(nn.Module):
    """Learned convolution in Fourier space: keep the lowest `modes` rfft coefficients,
    multiply by learned complex weights, zero the rest, transform back."""
    def __init__(self, in_ch, out_ch, modes):
        super().__init__()
        self.modes = modes
        scale = 1.0 / (in_ch * out_ch)
        self.weight = nn.Parameter(scale * torch.randn(in_ch, out_ch, modes, dtype=torch.cfloat))

    def forward(self, x):                       # x: (B, C, N)
        B, C, N = x.shape
        x_ft = torch.fft.rfft(x)                # (B, C, N//2 + 1)
        m = min(self.modes, x_ft.shape[-1])
        out_ft = torch.zeros(B, self.weight.shape[1], x_ft.shape[-1],
                             dtype=torch.cfloat, device=x.device)
        # (batch, in, mode) x (in, out, mode) -> (batch, out, mode)
        out_ft[..., :m] = torch.einsum("bim,iom->bom", x_ft[..., :m], self.weight[..., :m])
        return torch.fft.irfft(out_ft, n=N)

class FNO1d(nn.Module):
    """Lift -> [spectral conv + pointwise conv, GELU] x LAYERS -> project to 3 fields."""
    def __init__(self, in_ch=4, out_ch=3, width=64, modes=64, layers=4):
        super().__init__()
        self.lift = nn.Conv1d(in_ch, width, 1)
        self.spectral = nn.ModuleList([SpectralConv1d(width, width, modes) for _ in range(layers)])
        self.pointwise = nn.ModuleList([nn.Conv1d(width, width, 1) for _ in range(layers)])
        self.proj = nn.Sequential(nn.Conv1d(width, 128, 1), nn.GELU(), nn.Conv1d(128, out_ch, 1))

    def forward(self, x):                       # (B, in_ch, N) -> (B, out_ch, N)
        x = self.lift(x)
        for spec, pw in zip(self.spectral, self.pointwise):
            x = F.gelu(spec(x) + pw(x))
        return self.proj(x)

model = FNO1d(in_ch=4, out_ch=3, width=WIDTH, modes=MODES, layers=LAYERS).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"FNO1d: {n_params/1e6:.2f} M parameters")

## 4. Training

Plain MSE on the normalized targets, Adam + cosine LR schedule. We monitor the relative
$L^2$ error on the held-out solutions every few epochs (in normalized units — the
per-field physical errors come in the next section).

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

X_tr_d, Yn_tr_d = X_tr.to(DEVICE), Yn_tr.to(DEVICE)
X_te_d, Yn_te_d = X_te.to(DEVICE), Yn_te.to(DEVICE)
n_train = X_tr_d.shape[0]

def rel_l2(pred, true):
    return (torch.linalg.vector_norm(pred - true) / torch.linalg.vector_norm(true)).item()

history = {"train_mse": [], "test_rel_l2": []}
for epoch in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(n_train, device=DEVICE)
    running = 0.0
    for i in range(0, n_train, BATCH_SIZE):
        idx = perm[i:i + BATCH_SIZE]
        pred = model(X_tr_d[idx])
        loss = F.mse_loss(pred, Yn_tr_d[idx])
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        running += loss.item() * idx.numel()
    scheduler.step()
    history["train_mse"].append(running / n_train)

    if epoch % PRINT_EVERY == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            err = rel_l2(model(X_te_d), Yn_te_d)
        history["test_rel_l2"].append((epoch, err))
        print(f"epoch {epoch:4d}  train MSE {running / n_train:.4e}  test relL2 {err:.4e}")

## 5. Per-solution, per-field evaluation (physical units)

De-normalize and report the relative $L^2$ error of each field on each solution.
Expect: small errors on the training solutions and the interpolation tests (160, 200),
larger on 209 (mild extrapolation) and clearly larger on 131 (far outside the training
$\nu$-range) — a useful, honest picture of what the learned operator can and cannot do.

In [ ]:
@torch.no_grad()
def predict_physical(X):
    """Forward pass + de-normalization -> (B, 3, NX) numpy in physical units."""
    model.eval()
    out = model(X.to(DEVICE)).cpu().numpy()
    return out * sd_y + mu_y

def report(X, Y, sol_id, label):
    pred = predict_physical(X)
    print(f"--- {label} ---")
    for idx in sorted(set(sol_id.tolist())):
        m = sol_id == idx
        row = [f"V{idx} (nu={params[idx][2]:.3f}):"]
        for c, name in enumerate(["eta", "phi", "mu "]):
            num = np.linalg.norm(pred[m, c] - Y[m, c])
            den = np.linalg.norm(Y[m, c]) + 1e-14
            row.append(f"{name} {num/den:.3e}")
        print("  " + "   ".join(row))
    return pred

pred_tr = report(X_tr, Y_tr, id_tr, "training solutions")
pred_te = report(X_te, Y_te, id_te, "test solutions")



## 6. ID vs OOD: where does the learned operator break down?

**Split recap (REVERSED case): trained on large $\nu$, OOD = small $\nu$.** The ID
holdouts (V125, V150, V175, V200, V225) sit *inside* the trained large-amplitude side of
the family; the OOD block (V101–V110) is the small-amplitude gentle end, held out
entirely. Unlike the previous experiment, here the $\nu$-range test agrees with the
index-block definition: the OOD block has $\nu = 0.008$–$0.080$, cleanly below the
training minimum $\nu = 0.088$, so the shaded band in panel (a) and the OOD flag tell
the same story.

Three views:
1. **Error vs $\nu$** with the training range shaded (filled markers = ID, open = OOD).
   Expect the OOD points to sit left of the band; if this direction is easier than the
   steep-end extrapolation (as the physics suggests — smooth, near-sinusoidal targets),
   the OOD errors should be milder than in the previous experiment.
2. **Time-resolved $\eta$ error** over the quarter period (ID solid blue, OOD dashed red),
   normalized by each solution's overall $\eta$-norm (the instantaneous norm vanishes at
   $t=T/4$, where the standing wave is momentarily flat, so a pointwise relative error
   would spike artificially there).
3. **Snapshot contrast** at $t=T/4$: best ID vs worst OOD solution. Watch for the OOD
   failure mode of this direction: the operator dragging steep-family structure
   (harmonic content) into profiles that should be nearly pure cosine.

In [ ]:
# ---- Split the test solutions into ID vs OOD (REVERSED case) ----
# ID  = interior holdouts on the trained large-nu side -> interpolation.
# OOD = the held-out small-nu gentle end (V101-110)    -> extrapolation.
# Defined by the index block for consistency with the previous experiment;
# here the nu-range test agrees (OOD nu = 0.008-0.080 < training min 0.088).
nu_train = np.array([params[i][2] for i in TRAIN_INDICES])
nu_lo, nu_hi = nu_train.min(), nu_train.max()

test_ids = sorted(set(id_te.tolist()))
is_ood   = {i: i in TEST_OOD_INDICES for i in test_ids}
print(f"training nu-range: [{nu_lo:.3f}, {nu_hi:.3f}]")
for i in test_ids:
    print(f"  V{i}: nu = {params[i][2]:.3f}  ->  {'OOD (extrapolation)' if is_ood[i] else 'ID (interpolation)'}")


# ---- Per-solution, per-field relative L2 (aggregated over the quarter period) ----
def sol_field_err(idx):
    m = id_te == idx
    return [np.linalg.norm(pred_te[m, c] - Y_te[m, c]) /
            (np.linalg.norm(Y_te[m, c]) + 1e-14) for c in range(3)]
errs = {i: sol_field_err(i) for i in test_ids}

# ---- Compact summary table ----
print(f"\n{'solution':<10}{'nu':>8}{'regime':>8}" + "".join(f"{n:>12}" for n in ["eta", "phi", "mu"]))
for i in test_ids:
    tag = "OOD" if is_ood[i] else "ID"
    print(f"V{i:<9}{params[i][2]:>8.3f}{tag:>8}" + "".join(f"{e:>12.3e}" for e in errs[i]))

## 7. Export results to `.mat`

Everything needed to redo the figures in MATLAB without rerunning Python:
grids, truth and predictions for all held-out solutions as
`(n_test, Mt, 3, NX)` arrays (field order $\eta,\varphi,\mu$), the ID/OOD flag and
$\nu$/period per solution, per-solution per-field relative $L^2$ errors, and the
pooled ID/OOD errors per variable. Saved compressed, in float32 (~40 MB). The filename carries the experiment tag
`ood_smallnu` to distinguish it from the previous case's `fno_wave_results.mat`
(train small $\nu$ / OOD large $\nu$), so both result files can coexist in `DATA_DIR`.

In [ ]:
# =========================
# Save truth, predictions, and errors to .mat for later (MATLAB) visualization
# =========================
from scipy.io import savemat

test_ids_sorted = TEST_ID_INDICES + TEST_OOD_INDICES     # ID block first, then OOD
n_test = len(test_ids_sorted)
Mt = int((id_te == test_ids_sorted[0]).sum())            # timeslices per solution
tau = np.linspace(0.0, 1.0, 201)[::TIME_STRIDE]          # shared normalized time grid

# Reshape flat test arrays into per-solution blocks: (n_test, Mt, 3, NX)
Y_true_mat = np.zeros((n_test, Mt, 3, NX), dtype=np.float32)
Y_pred_mat = np.zeros((n_test, Mt, 3, NX), dtype=np.float32)
for k, i in enumerate(test_ids_sorted):
    m = id_te == i
    Y_true_mat[k] = Y_te[m]
    Y_pred_mat[k] = pred_te[m]

# Per-solution, per-field relative L2 errors, aligned with test_ids_sorted
err_per_sol = np.array([[np.linalg.norm(Y_pred_mat[k, :, c] - Y_true_mat[k, :, c]) /
                         (np.linalg.norm(Y_true_mat[k, :, c]) + 1e-14)
                         for c in range(3)] for k in range(n_test)])

# Pooled ID / OOD errors per field (rows: eta, phi, mu; plus overall)
ood_flag = np.array([i in TEST_OOD_INDICES for i in test_ids_sorted])
def pooled(sel, c=None):
    A, B = Y_pred_mat[sel], Y_true_mat[sel]
    if c is not None:
        A, B = A[:, :, c], B[:, :, c]
    return np.linalg.norm(A - B) / (np.linalg.norm(B) + 1e-14)
err_pooled = np.array([[pooled(~ood_flag, c), pooled(ood_flag, c)] for c in range(3)]
                      + [[pooled(~ood_flag), pooled(ood_flag)]])   # (4, 2): rows eta/phi/mu/all, cols ID/OOD

# Experiment-tagged filename: this is the REVERSED case (train large nu, OOD small
# nu). The previous case saved to "fno_wave_results.mat"; keeping distinct names
# lets both experiments' results live side by side for the comparison plots.
mat_path = os.path.join(MAT_DIR, "fno_wave_results_ood_smallnu.mat")
savemat(mat_path, {
    # grids
    "x_grid": x_grid,                                  # (NX,) mirrored grid on (-pi, pi]
    "tau": tau,                                        # (Mt,) normalized time t/(T/4)
    # fields: (n_test, Mt, 3, NX), field order eta | phi | mu
    "Y_true": Y_true_mat,
    "Y_pred": Y_pred_mat,
    # per-solution metadata, aligned with the first dimension above
    "test_indices": np.asarray(test_ids_sorted, dtype=np.int32),
    "is_ood": ood_flag.astype(np.int32),               # 0 = ID, 1 = OOD
    "nu": np.array([params[i][2] for i in test_ids_sorted]),        # wave height
    "periods": np.array([params[i][1] for i in test_ids_sorted]),   # physical T
    "max_slope": np.array([params[i][4] for i in test_ids_sorted]), # steepness
    # errors
    "err_per_sol": err_per_sol,                        # (n_test, 3) rel L2 per field
    "err_pooled": err_pooled,                          # (4, 2) rows eta/phi/mu/all, cols ID/OOD
    # provenance
    "train_indices": np.asarray(TRAIN_INDICES, dtype=np.int32),
    "config": dict(modes=MODES, width=WIDTH, layers=LAYERS,
                   time_stride=TIME_STRIDE, epochs=EPOCHS, seed=SEED),
}, do_compression=True)
print("saved:", mat_path)
print("Y_true/Y_pred shape:", Y_true_mat.shape, " (n_test, Mt, fields, NX)")
print("pooled rel L2 [rows eta/phi/mu/all; cols ID, OOD]:")
print(np.array2string(err_pooled, formatter={'float_kind': lambda v: f'{v:.3e}'}))

## 8. Save the trained operator

Weights plus the normalization statistics and grid — everything needed to reload and
evaluate the operator on any $\eta_0(x)$ later (e.g., from other family members, or as a
surrogate inside a data-assimilation loop).

In [ ]:
# Experiment-tagged checkpoint (REVERSED case: train large nu, OOD small nu);
# the previous case's operator remains at "fno_wave_operator.pt".
ckpt_path = os.path.join(OUT_DIR, "checkpoints", "fno_wave_operator_ood_smallnu.pt")
torch.save({
    "model_state": model.state_dict(),
    "mu_y": mu_y, "sd_y": sd_y,
    "x_grid": x_grid,
    "config": dict(modes=MODES, width=WIDTH, layers=LAYERS,
                   train_indices=TRAIN_INDICES, time_stride=TIME_STRIDE),
}, ckpt_path)
print("saved:", ckpt_path)

In [ ]:
# =========================
# Aggregated ID vs OOD errors per variable
# =========================
# Pool all timeslices of all ID solutions together (same for OOD) and compute
# one relative L2 error per field. This gives the two headline numbers per
# variable: interpolation accuracy vs extrapolation accuracy.

FIELDS = ["eta", "phi", "mu"]

# Boolean masks over the test samples, based on the automatic split of Section 7
mask_id  = np.array([not is_ood[i] for i in id_te])   # ID  = interior holdouts (large-nu side)
mask_ood = np.array([is_ood[i]     for i in id_te])   # OOD = held-out small-nu gentle end

def pooled_rel_l2(mask, c):
    """Relative L2 error of field channel c, pooled over all samples in `mask`."""
    num = np.linalg.norm(pred_te[mask, c] - Y_te[mask, c])
    den = np.linalg.norm(Y_te[mask, c]) + 1e-14
    return num / den

print(f"{'variable':<10}{'ID error':>14}{'OOD error':>14}{'ratio (OOD/ID)':>18}")
for c, name in enumerate(FIELDS):
    e_id  = pooled_rel_l2(mask_id,  c)
    e_ood = pooled_rel_l2(mask_ood, c)
    print(f"{name:<10}{e_id:>14.4e}{e_ood:>14.4e}{e_ood / e_id:>18.2f}")

# Overall (all three variables pooled)
e_id_all  = np.linalg.norm(pred_te[mask_id]  - Y_te[mask_id])  / (np.linalg.norm(Y_te[mask_id])  + 1e-14)
e_ood_all = np.linalg.norm(pred_te[mask_ood] - Y_te[mask_ood]) / (np.linalg.norm(Y_te[mask_ood]) + 1e-14)
print(f"{'all':<10}{e_id_all:>14.4e}{e_ood_all:>14.4e}{e_ood_all / e_id_all:>18.2f}")

## 9. Exploratory: even smaller ε, beyond JW's data
The dataset ends at ε = 0.0123 (V240). Here we push the trained operator to hypothetical
waves at even smaller ε — **not an OOD test** (no ground truth exists), just exploratory
predictions. Since the FNO's input is the initial profile η₀(x), not ε itself, we first
construct η₀ at the target ε by **pointwise quadratic extrapolation in ε** over
the steepest TRAINED solutions (V231-240), then run the operator. Sanity prints: the extrapolated profiles' crest height
and max slope (they should continue the sharper-not-taller trend), and the operator's
self-consistency at τ = 0 (its η output should reproduce the input profile). Results are
saved to `figures-matlab/` in the same layout as the main export (with `Y_true` absent and
an `is_exploratory` flag).

In [ ]:
# =========================
# 11. Try even smaller epsilon (beyond the end of JW's branch data)
# =========================
EPS_TRY = [0.010, 0.008, 0.006]          # hypothetical epsilon values, all below 0.0123
N_FIT   = 10                             # steepest solutions used for the extrapolation
FIT_DEG = 2                              # polynomial degree in epsilon (pointwise in x)

# --- steepest available solutions (V231..V240 -- in this experiment these are trained solutions) ---
P_par = np.loadtxt(os.path.join(DATA_DIR, "parameters"))
def eps_of(i): return 1.0 - P_par[P_par[:, 0] == i, 3][0]
fit_ids  = list(range(241 - N_FIT, 241))
fit_eps  = np.array([eps_of(i) for i in fit_ids])
fit_eta0 = np.stack([mirror_extend(load_V(os.path.join(DATA_DIR, f"V{i}"))[0])[0]
                     for i in fit_ids])                  # (N_FIT, 512) eta at t = 0

# --- pointwise quadratic extrapolation of eta0(x; eps) to the target eps ---
coef = np.polyfit(fit_eps, fit_eta0, FIT_DEG)            # (deg+1, 512)
eta0_try = np.stack([np.polyval(coef, e) for e in EPS_TRY]).astype(np.float32)

# sanity: crest height and max slope should continue the sharper-not-taller trend
kw = np.fft.rfftfreq(NX, d=(x_grid[1] - x_grid[0])) * 2 * np.pi
for j, e in enumerate(EPS_TRY):
    sl = np.abs(np.fft.irfft(1j * kw * np.fft.rfft(eta0_try[j]), n=NX)).max()
    print(f"eps = {e:.3f}: extrapolated crest height {eta0_try[j].max():.4f}, max slope {sl:.3f}")

# --- reuse the notebook's own pipeline: pack_inputs -> predict_physical ---
t_full = np.linspace(0.0, 1.0, 201)[::TIME_STRIDE]
n_try, Mt = len(EPS_TRY), len(t_full)
X0_try = np.repeat(eta0_try, Mt, axis=0)                 # (n_try*Mt, 512)
Tn_try = np.tile(t_full.astype(np.float32), n_try)
Yp_try = predict_physical(pack_inputs(X0_try, Tn_try)).reshape(n_try, Mt, 3, NX)

# self-consistency at tau = 0: predicted eta should reproduce the input profile
for j, e in enumerate(EPS_TRY):
    sc = np.linalg.norm(Yp_try[j, 0, 0] - eta0_try[j]) / np.linalg.norm(eta0_try[j])
    print(f"eps = {e:.3f}: tau=0 self-consistency rel error = {sc:.2e}")

# --- save in the same layout as the main export (no Y_true; flagged exploratory) ---
savemat(os.path.join(MAT_DIR, "fno_wave_results_ood_smallnu_eps_try.mat"), {
    "Y_pred": Yp_try.astype(np.float32),          # (n_try, n_tau, 3, 512)
    "eta0_try": eta0_try,                         # extrapolated initial profiles
    "eps_try": np.asarray(EPS_TRY, dtype=np.float32),
    "x_grid": x_grid.astype(np.float32),
    "tau": t_full.astype(np.float32),
    "fit_indices": np.asarray(fit_ids, dtype=np.int32),
    "fit_degree": FIT_DEG,
    "is_exploratory": 1,                          # NO ground truth: not an OOD test
}, do_compression=True)
print("saved exploratory results to results/fno_wave_results_ood_smallnu_eps_try.mat")